# Experiment 3: Ablation Studies
**Dataset:** Pascal VOC, 100 epochs per ablation.
| # | Ablation | What Changes |
|---|----------|--------------|
| A4 | ReLU6 vs SiLU | Activation |
| A6 | Resolution sweep | 224,320,416,640 |
| A9 | Mosaic on vs off | Augmentation |

In [ ]:
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q && pip install tqdm -q
import torch, os, json, glob
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')" 2>/dev/null

In [ ]:
print('A4: ReLU6 (quantized)...')
get_ipython().system('python scripts/train.py --task det --variant quantized --data voc.yaml --imgsz 416 --epochs 100 --seed 42 --warmup 3 --name ablation-a4-relu6')
print('\nA4: SiLU (standard)...')
get_ipython().system('python scripts/train.py --task det --variant standard --data voc.yaml --imgsz 416 --epochs 100 --seed 42 --warmup 3 --name ablation-a4-silu')
print('✅ A4 complete')

In [ ]:
for res in [224, 320, 416, 640]:
    print(f'\nA6: {res}x{res}...')
    get_ipython().system(f'python scripts/train.py --task det --variant quantized --data voc.yaml --imgsz {res} --epochs 100 --seed 42 --warmup 3 --name ablation-a6-{res}')
print('✅ A6 complete')

In [ ]:
os.environ['TINYYOLO_NO_MOSAIC'] = '1'
get_ipython().system('python scripts/train.py --task det --variant quantized --data voc.yaml --imgsz 416 --epochs 100 --seed 42 --warmup 3 --name ablation-a9-no-mosaic')
del os.environ['TINYYOLO_NO_MOSAIC']
print('✅ A9 complete')

In [ ]:
print('=' * 80)
print('  Ablation Results Summary')
print('=' * 80)
ablation_data = []
for f in sorted(glob.glob('experiments/results/ablation-*/config.json')):
    with open(f) as fh: cfg = json.load(fh)
    fm = cfg.get('final_metrics', {})
    name = Path(f).parent.name
    row = {'experiment': name, 'params_M': cfg.get('params_M', 0), 'mAP50': fm.get('mAP50', 0), 'mAP50_95': fm.get('mAP50_95', fm.get('mAP50-95', 0))}
    ablation_data.append(row)
    print(f'  {name:35s}: mAP@50={row["mAP50"]:.4f}  params={row["params_M"]:.3f}M')
with open('experiments/results/ablation_summary.json', 'w') as f: json.dump(ablation_data, f, indent=2)
print(f'Saved {len(ablation_data)} results')

In [ ]:
!cd experiments && zip -r /content/ablation_results.zip results/ablation-*
print('📥 Download: /content/ablation_results.zip')